# Baseline: NER on Clean CoNLL-2003

Evaluate three tokenizers on clean CoNLL-2003 NER data.  
This is **Phase 1** of the case study — the reference point before adding noise.

**Tokenizers:**
- `bert-base-uncased` — WordPiece
- `gpt2` — Byte Pair Encoding (BPE)
- `google/byt5-small` — Byte-level (no vocabulary, no OOV)

**Metrics:**
- Tokenizer: fertility (tokens/word), OOV rate, vocab coverage
- NER: seqeval entity-level F1 (standard CoNLL metric)


## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

from metrics import compute_tokenizer_stats, print_stats_table
from train import get_model_and_tokenizer, tokenize_and_align_labels, make_compute_metrics

# ── Device ─────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

# ── Config ─────────────────────────────────────────────────────────────────────
MODEL_NAMES = [
    "bert-base-uncased",
    "gpt2",
    "google/byt5-small",
]

RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), "results", "tables")
os.makedirs(RESULTS_DIR, exist_ok=True)


## 1. Load CoNLL-2003

Standard English NER benchmark. Labels: `O`, `B/I-PER`, `B/I-ORG`, `B/I-LOC`, `B/I-MISC`


In [ ]:
dataset = load_dataset("conll2003", trust_remote_code=True)

label_names = dataset["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
num_labels = len(label_names)

print(f"Labels ({num_labels}): {label_names}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")


In [ ]:
# Preview a few examples
for ex in dataset["train"].select(range(3)):
    tokens = ex["tokens"]
    tags = [label_names[t] for t in ex["ner_tags"]]
    print(" | ".join(f"{tok}/{tag}" for tok, tag in zip(tokens, tags)))
    print()


## 2. Tokenizer Analysis

Measure how each tokenizer handles clean CoNLL-2003 text.

- **Fertility** — avg tokens produced per word (higher = more fragmentation)
- **OOV rate** — fraction of words producing at least one UNK token
- **UNK rate** — fraction of all tokens that are UNK


In [ ]:
from transformers import AutoTokenizer

sentences = [ex["tokens"] for ex in dataset["train"]]

tokenizer_stats = {}
for model_name in MODEL_NAMES:
    print(f"Analyzing {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    stats = compute_tokenizer_stats(tok, sentences, sample_size=3000)
    tokenizer_stats[model_name] = stats

print()
print_stats_table(tokenizer_stats)


In [ ]:
# Save tokenizer stats to CSV
stats_df = pd.DataFrame(tokenizer_stats).T.reset_index()
stats_df.columns = ["model", "fertility", "oov_rate", "unk_rate", "vocab_size"]
stats_df.to_csv(os.path.join(RESULTS_DIR, "tokenizer_stats_clean.csv"), index=False)
print("Saved tokenizer_stats_clean.csv")
stats_df


## 3. NER Fine-tuning

Train each model on CoNLL-2003 train set, evaluate on test set.

> **Note:** Training all three models takes ~30–90 min on MPS depending on hardware.
> You can reduce `num_train_epochs` for quick iteration.


### 3.1 BERT (WordPiece)

In [ ]:
MODEL_NAME = "bert-base-uncased"
model, tokenizer = get_model_and_tokenizer(MODEL_NAME, num_labels, id2label, label2id)

tokenized_bert = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, tokenizer),
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("Tokenized. Example token count:", len(tokenized_bert["train"][0]["input_ids"]))


In [ ]:
training_args = TrainingArguments(
    output_dir=f"../results/checkpoints/{MODEL_NAME.replace('/', '_')}",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer_bert = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_bert["train"],
    eval_dataset=tokenized_bert["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=make_compute_metrics(label_names),
)

trainer_bert.train()


In [ ]:
bert_results = trainer_bert.evaluate(tokenized_bert["test"])
print(f"BERT  |  F1: {bert_results['eval_f1']:.4f}  |  "
      f"Precision: {bert_results['eval_precision']:.4f}  |  "
      f"Recall: {bert_results['eval_recall']:.4f}")


### 3.2 GPT-2 (BPE)

In [ ]:
MODEL_NAME = "gpt2"
model_gpt2, tokenizer_gpt2 = get_model_and_tokenizer(MODEL_NAME, num_labels, id2label, label2id)

tokenized_gpt2 = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, tokenizer_gpt2),
    batched=True,
    remove_columns=dataset["train"].column_names,
)


In [ ]:
training_args_gpt2 = TrainingArguments(
    output_dir=f"../results/checkpoints/gpt2",
    num_train_epochs=3,
    per_device_train_batch_size=8,   # GPT-2 uses more memory
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer_gpt2 = Trainer(
    model=model_gpt2,
    args=training_args_gpt2,
    train_dataset=tokenized_gpt2["train"],
    eval_dataset=tokenized_gpt2["validation"],
    tokenizer=tokenizer_gpt2,
    data_collator=DataCollatorForTokenClassification(tokenizer_gpt2),
    compute_metrics=make_compute_metrics(label_names),
)

trainer_gpt2.train()


In [ ]:
gpt2_results = trainer_gpt2.evaluate(tokenized_gpt2["test"])
print(f"GPT-2 |  F1: {gpt2_results['eval_f1']:.4f}  |  "
      f"Precision: {gpt2_results['eval_precision']:.4f}  |  "
      f"Recall: {gpt2_results['eval_recall']:.4f}")


### 3.3 ByT5 (Byte-level)

ByT5 tokenizes text into individual bytes — no vocabulary needed, zero OOV by definition.
We use the T5 encoder + linear classification head (`ByT5ForTokenClassification`).

> ByT5 produces ~3-4x more tokens per sentence than WordPiece, so we reduce batch size.


In [ ]:
MODEL_NAME = "google/byt5-small"
model_byt5, tokenizer_byt5 = get_model_and_tokenizer(MODEL_NAME, num_labels, id2label, label2id)

tokenized_byt5 = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, tokenizer_byt5, max_length=1024),
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("Tokenized. Example token count:", len(tokenized_byt5["train"][0]["input_ids"]))


In [ ]:
training_args_byt5 = TrainingArguments(
    output_dir=f"../results/checkpoints/byt5",
    num_train_epochs=3,
    per_device_train_batch_size=4,   # byte-level = long sequences
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=200,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer_byt5 = Trainer(
    model=model_byt5,
    args=training_args_byt5,
    train_dataset=tokenized_byt5["train"],
    eval_dataset=tokenized_byt5["validation"],
    tokenizer=tokenizer_byt5,
    data_collator=DataCollatorForTokenClassification(tokenizer_byt5, pad_to_multiple_of=8),
    compute_metrics=make_compute_metrics(label_names),
)

trainer_byt5.train()


In [ ]:
byt5_results = trainer_byt5.evaluate(tokenized_byt5["test"])
print(f"ByT5  |  F1: {byt5_results['eval_f1']:.4f}  |  "
      f"Precision: {byt5_results['eval_precision']:.4f}  |  "
      f"Recall: {byt5_results['eval_recall']:.4f}")


## 4. Baseline Results Summary

Reference F1 scores on **clean** CoNLL-2003 test set.
This table will be extended in notebook `03_noisy_evaluation.ipynb`.


In [ ]:
summary = pd.DataFrame([
    {
        "model": "bert-base-uncased",
        "strategy": "WordPiece",
        **tokenizer_stats["bert-base-uncased"],
        "f1_clean": bert_results["eval_f1"],
        "precision_clean": bert_results["eval_precision"],
        "recall_clean": bert_results["eval_recall"],
    },
    {
        "model": "gpt2",
        "strategy": "BPE",
        **tokenizer_stats["gpt2"],
        "f1_clean": gpt2_results["eval_f1"],
        "precision_clean": gpt2_results["eval_precision"],
        "recall_clean": gpt2_results["eval_recall"],
    },
    {
        "model": "google/byt5-small",
        "strategy": "Byte-level",
        **tokenizer_stats["google/byt5-small"],
        "f1_clean": byt5_results["eval_f1"],
        "precision_clean": byt5_results["eval_precision"],
        "recall_clean": byt5_results["eval_recall"],
    },
])

summary.to_csv(os.path.join(RESULTS_DIR, "baseline_results.csv"), index=False)
print("Saved baseline_results.csv")
summary
